# P2-07, P2-08 and P2-09
Run the cells in order on a CPU runtime. The inspection checks all audio headers and may take several hours on Drive. See docs/p2_07_09_execution.md for recovery and interpretation.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import sys, os, json, subprocess
PROJECT_ROOT = Path("/content/drive/MyDrive/PAI&AIES Project")
assert (PROJECT_ROOT / "pyproject.toml").is_file(), "Set PROJECT_ROOT to your project"
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
SNAPSHOT = PROJECT_ROOT / "data/manifests/phase2_raw/20261101T170035Z-7aac9dc154ad"
PAIRING = PROJECT_ROOT / "data/manifests/phase2_pairs/20261101T211532Z-ca15b910a412"
RESUME_P207 = ""  # Set the printed run ID after an interrupted run.


## Dependencies and fixture checks
The cell stops if a required fixture check is skipped. No research audio is read here.


In [ ]:
import yaml, pyarrow, soundfile, numpy
from echoproof.phase2_config import load_phase2_config
from echoproof.data.phase2_quality import inspect, reconcile, capacity
CONFIG = load_phase2_config(PROJECT_ROOT / "configs/phase2.yaml", PROJECT_ROOT)
check = subprocess.run([sys.executable,"-m","unittest","discover","-s",str(PROJECT_ROOT/"tests"),"-p","test_p2_07_09_quality.py","-v"],cwd=PROJECT_ROOT,text=True,capture_output=True)
print(check.stdout, check.stderr)
assert check.returncode == 0 and "skipped" not in (check.stdout + check.stderr).lower(), "Resolve failed or skipped checks"


## P2-07: inspect headers and sampled audio
Save the printed run ID. A completed or attention-needed inspection produces inspection_summary.json and admission.json.


In [ ]:
inspection = inspect(PROJECT_ROOT, CONFIG, SNAPSHOT, PAIRING, resume=RESUME_P207 or None)
INSPECTION = Path(inspection["output"])
print("P2-07 output:", INSPECTION)
print(json.dumps(inspection,indent=2))


## P2-08: reconcile counts and generate the source card
If reconnecting in a new runtime, set INSPECTION to the completed P2-07 run directory before running this cell.


In [ ]:
# INSPECTION = PROJECT_ROOT / "data/manifests/phase2_inspection/<completed-run-id>"
reconciliation = reconcile(PROJECT_ROOT, CONFIG, SNAPSHOT, PAIRING, INSPECTION)
print("P2-08 output:",reconciliation["output"])
print(json.dumps(reconciliation,indent=2))


## P2-09: capacity record
Enter current account values from Google Drive storage, in decimal GB. Use None for an unknown value; the output will remain pending. The scratch measurement is separate from Drive account quota. No acquisition is started.


In [ ]:
DRIVE_TOTAL_GB = None
DRIVE_USED_GB = None
DRIVE_FREE_GB = None
REMAINING_WORK_HOURS = None
# Set these only for a specific planned acquisition, otherwise keep zero.
PLANNED_DOWNLOAD_BYTES = 0
EXPANDED_BYTES = 0
STAGING_BYTES = 0
CACHE_BYTES = 0
budget = capacity(PROJECT_ROOT, drive_total_gb=DRIVE_TOTAL_GB, drive_used_gb=DRIVE_USED_GB, drive_free_gb=DRIVE_FREE_GB, remaining_hours=REMAINING_WORK_HOURS, scratch_path="/content", planned_download_bytes=PLANNED_DOWNLOAD_BYTES, expanded_bytes=EXPANDED_BYTES, staging_bytes=STAGING_BYTES, cache_bytes=CACHE_BYTES)
print("P2-09 output:",budget["output"])
print(json.dumps(budget,indent=2))
